# 03 - Meta-learners: T-Learner and X-Learner

Two meta-learners that estimate the conditional average treatment effect
(CATE), `tau(x) = E[Y(1) - Y(0) | X = x]`, rather than response probability.

Run `01_data_processing.ipynb` first.

In [ ]:
import sys
from pathlib import Path


def _find_repo_root() -> Path:
    """Locate the repo root without assuming the working directory.

    A fresh Kaggle kernel starts in /kaggle/working, not in the repo, so we
    search: the current directory and its parents (local development), then
    /kaggle/working and each attached /kaggle/input/<slug>/ (Kaggle, where
    the repo is cloned into working or attached as a dataset).
    """
    bases = [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working"), Path("/kaggle/input")]
    for base in bases:
        if not base.is_dir():
            continue
        if (base / "src" / "data.py").is_file():
            return base
        for child in sorted(p for p in base.iterdir() if p.is_dir()):
            if (child / "src" / "data.py").is_file():
                return child
    raise RuntimeError(
        "Could not locate the repository root (no src/data.py found). On Kaggle, "
        "clone this repository into /kaggle/working or attach it as a dataset."
    )


REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("repo root:", REPO_ROOT)

In [ ]:
from src.data import PRIMARY_OUTCOME, TREATMENT_COLUMN, load_config, load_parquet, output_dir, save_parquet
from src.preprocessing import LightGBMFeatureTransform
from src.models import fit_t_learner, fit_x_learner
from src.evaluation import evaluate_ranking

CONFIG = load_config()
OUTPUT_DIR = output_dir()

train_frame = load_parquet(OUTPUT_DIR / "train.parquet")
val_frame = load_parquet(OUTPUT_DIR / "validation.parquet")
test_frame = load_parquet(OUTPUT_DIR / "test.parquet")

transform = LightGBMFeatureTransform().fit(train_frame)
X_train, X_val, X_test = (transform.transform(f) for f in (train_frame, val_frame, test_frame))
Y_train, Y_val, Y_test = (f[PRIMARY_OUTCOME] for f in (train_frame, val_frame, test_frame))
T_train, T_val, T_test = (f[TREATMENT_COLUMN] for f in (train_frame, val_frame, test_frame))
train_frame.shape, val_frame.shape, test_frame.shape

## T-Learner

Two independent outcome models, one per arm:
`mu1(x) = E[Y | T=1, X=x]`, `mu0(x) = E[Y | T=0, X=x]`, and
`tau(x) = mu1(x) - mu0(x)`.

The simplest CATE estimator, and the baseline the others are compared to.
Its weakness on this dataset is arm imbalance: the control arm is far
smaller, so `mu0` is fit on much less data than `mu1`.

In [ ]:
t_learner = fit_t_learner(X_train, T_train, Y_train, X_val, T_val, Y_val, seed=CONFIG["seed"])
print("mu1 best iteration:", t_learner.mu1.best_iteration)
print("mu0 best iteration:", t_learner.mu0.best_iteration)

In [ ]:
t_val_scores = t_learner.predict_tau(X_val)
t_val_ranking = evaluate_ranking(t_val_scores, T_val, Y_val)
print("T-Learner validation qini_above_random:", round(t_val_ranking.qini_above_random, 4))
print("T-Learner validation auuc_above_random:", round(t_val_ranking.auuc_above_random, 4))
t_val_ranking.uplift_at_k

## X-Learner

Designed for exactly the arm imbalance the T-Learner struggles with. Two
stages:

1. **Nuisance stage** -- `mu1`, `mu0` fit with two-fold cross-fitting. Every
   row's out-of-fold prediction comes from the model fit on the *opposite*
   fold.
2. **Effect stage** -- pseudo-outcomes `D1 = Y - mu0_oof` (treated rows) and
   `D0 = mu1_oof - Y` (control rows), each regressed on `X` to give `tau1`,
   `tau0`, combined by the treatment rate.

**Leakage note.** Two things must be fold-local here, not one. The nuisance
*models* obviously, but also the *categorical vocabulary* each model is
trained under: fitting one transform on the whole train partition and
reusing it for both folds lets each fold's vocabulary be informed by the
opposite fold. `fit_x_learner` therefore takes the **raw** frame and fits
its own fold-local transforms internally, and raises if handed an
already-transformed frame.

In [ ]:
x_learner = fit_x_learner(train_frame, T_train, Y_train, seed=CONFIG["seed"])
x_val_scores = x_learner.predict_tau(X_val)
x_val_ranking = evaluate_ranking(x_val_scores, T_val, Y_val)
print("X-Learner validation qini_above_random:", round(x_val_ranking.qini_above_random, 4))
print("X-Learner validation auuc_above_random:", round(x_val_ranking.auuc_above_random, 4))
x_val_ranking.uplift_at_k

## Compare (validation)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for label, r in [("T-Learner", t_val_ranking), ("X-Learner", x_val_ranking)]:
    ax1.plot(r.qini_curve["coverage"], r.qini_curve["qini_gain"], label=label)
    ax2.plot(r.uplift_curve["coverage"], r.uplift_curve["uplift_gain"], label=label)
ax1.plot([0, 1], [0, t_val_ranking.theoretical_random_qini_area * 2], "--", label="Theoretical random")
ax2.plot([0, 1], [0, t_val_ranking.theoretical_random_auuc_area * 2], "--", label="Theoretical random")
ax1.set_xlabel("Coverage"); ax1.set_ylabel("Qini gain"); ax1.set_title("Qini curve"); ax1.legend()
ax2.set_xlabel("Coverage"); ax2.set_ylabel("Uplift gain"); ax2.set_title("Uplift curve (AUUC)"); ax2.legend()
fig.tight_layout()

## Save test predictions

In [ ]:
import pandas as pd

for name, scores in [("tlearner", t_learner.predict_tau(X_test)), ("xlearner", x_learner.predict_tau(X_test))]:
    frame = pd.DataFrame({
        "score": scores,
        TREATMENT_COLUMN: T_test.to_numpy(),
        PRIMARY_OUTCOME: Y_test.to_numpy(),
    })
    save_parquet(frame, OUTPUT_DIR / f"preds_{name}_test.parquet")
    print("saved ->", OUTPUT_DIR / f"preds_{name}_test.parquet")

## Next

Continue with `04_causal_forest.ipynb`.